In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import json
import random
from itertools import combinations

# Dependency:
* DDBC
* DDBC_MSIGDB

# Loading variables

In [ ]:
DISEASE = "BIPOLAR"
OUTPUT_DIRECTORY = f"../output/{DISEASE}/"
NONE_OUTPUT_DIRECTORY  = f"../output/NONE/"
NULL_OUTPUT_DIRECTORY = f"../output/NULL/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = f"../../Gen_Hypergraph/output/MSIGDB_FULL/"

In [ ]:
with open(f"{DGIDB_DIRECTORY}/gene_to_index.json", "rb") as f:
    dgidb_gene_to_index = json.load(f)
with open(f"{MSIGDB_DIRECTORY}/gene_to_index.json", "rb") as f:
    msigdb_gene_to_index = json.load(f)
with open(f"{OUTPUT_DIRECTORY}/gene_to_index_distinct.json", "rb") as f:
    gene_to_index_distinct = json.load(f)
with open(f"{OUTPUT_DIRECTORY}/dgidb_to_msigdb_indices_dict.json", "rb") as f:
    dgidb_to_msigdb_dgidb_indices_dict = json.load(f)

In [ ]:
num_distinct_genes = len(gene_to_index_distinct)
num_genes_msigdb = len(msigdb_gene_to_index)
num_additional_rows = num_distinct_genes - num_genes_msigdb

In [ ]:
print(num_distinct_genes,num_genes_msigdb,num_additional_rows)

In [ ]:
# ## Sanity Check
# index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}
# dgidb_index_to_gene = {v: k for k, v in dgidb_gene_to_index.items()}
# msigdb_index_to_gene = {v: k for k, v in msigdb_gene_to_index.items()}

# for i in range(len(indices_agg)):
#     gene_name = index_to_gene_distinct[indices_agg[i]]
#     p = dgidb_gene_to_index.get(gene_name, None)
#     assert p is not None

# for i in range(len(indices)):
#     gene_name = msigdb_index_to_gene[indices[i]]
#     p = dgidb_gene_to_index.get(gene_name, None)
#     assert p is not None

In [ ]:
# load ddm
ddm_both = np.load(f"{OUTPUT_DIRECTORY}/ddm_[2, 4, 6, 8].npy")

In [ ]:
ddm_both.mean()

In [ ]:
ddm_both_cropped = ddm_both[num_additional_rows:, num_additional_rows:]

In [ ]:
# load npy arrays for NONE
ddm_msigdb = np.load(f"{NONE_OUTPUT_DIRECTORY}/ddm_[2, 4, 6, 8].npy")

In [ ]:
ddm_msigdb.mean()

In [ ]:
# load ddm for the null model (multilayer, with a degree-preserving randomized DGIdb layer).
# mmap_mode keeps this off the heap: the matrix is ~3.9 GB and only row averages are needed.
run_id = "20260729_220027"
ddm_null = np.load(f"{NULL_OUTPUT_DIRECTORY}/ddm_[2, 4, 6, 8]_{run_id}.npy", mmap_mode="r")

# The null model rewires edges but never renames genes, so its distinct-gene ordering must
# match the real multilayer one for a row-by-row comparison to mean anything. The NULL run
# was seeded from BIPOLAR, so this only lines up when DISEASE == "BIPOLAR".
with open(f"{NULL_OUTPUT_DIRECTORY}/gene_to_index_distinct.json", "rb") as f:
    gene_to_index_distinct_null = json.load(f)
assert gene_to_index_distinct_null == gene_to_index_distinct, (
    f"NULL gene ordering does not match BIPOLAR; the null model was built from a "
    "different disease layer, so the rows are not comparable."
)
assert ddm_null.shape == ddm_both.shape

In [ ]:
ddm_null.mean()

In [ ]:
dgidb_indices = [x for x in list(dgidb_to_msigdb_indices_dict.values()) if x is not None]
dgidb_indices_agg = [x + num_additional_rows for x in dgidb_indices]

# New Fig 4

In [ ]:
# num_samples = int(1e6)

In [ ]:
# # Random Sample
# distance_single = []
# distance_both = []
# for _ in range(num_samples):
#     i, j = random.sample(range(num_genes_msigdb), 2)
#     d_single = ddm_msigdb[i,j]
#     d_both = ddm_both_cropped[i,j]
#     distance_single.append(d_single)
#     distance_both.append(d_both)

In [ ]:
# # DGIDB Pairwise Only
# distance_single = []
# distance_both = []
# for i,j in combinations(indices,2):
#     d_single = ddm_msigdb[i,j]
#     d_both = ddm_both_cropped[i,j]
#     distance_single.append(d_single)
#     distance_both.append(d_both)

In [ ]:
# plt.scatter(distance_single,distance_both)

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 8))

# # --- THE BACKGROUND (Choose Hexbin OR KDE) ---

# # Choice 1: Hexbin (Best if you have millions of points; very fast)
# # 'bins=log' is crucial here so the few dense areas don't wash out the rest
# hb = ax.hexbin(distance_single, distance_both, gridsize=60, cmap='Greys',
#                mincnt=1, bins='log', alpha=0.7)
# cb = fig.colorbar(hb, ax=ax, label='$\log_{10}(N)$ Gene Pairs')

# # Choice 2: Seaborn KDE (Uncomment below if you prefer smooth topographic lines)
# # Note: KDE can be slow to compute if you feed it millions of rows.
# # sns.kdeplot(x=single_layer_bg, y=multi_layer_bg, cmap="Greys", fill=True,
# #             thresh=0.05, levels=10, ax=ax, alpha=0.7)

# # --- THE REFERENCE LINE ---
# # Draw the y=x line to show where points *would* be if nothing changed
# ax.plot([0, 0.035], [0, 0.035], color='black', linestyle='--', linewidth=1.5,
#         alpha=0.8, zorder=3, label='Baseline ($y=x$)')

# # ==========================================
# # 3. FORMATTING FOR PUBLICATION
# # ==========================================
# ax.set_xlabel('Single-Layer Diffusion Distance', fontsize=12, fontweight='bold')
# ax.set_ylabel('Multi-Layer Diffusion Distance', fontsize=12, fontweight='bold')
# ax.set_title('Diffusion Geometry Reshaping: The "Wormhole" Effect', fontsize=14, fontweight='bold')

# # Clean up axes limits based on your actual data
# ax.set_xlim(0, 0.035)
# ax.set_ylim(0, 0.035)

# ax.legend(fontsize=11, loc='upper left')
# ax.grid(True, linestyle=':', alpha=0.5)

# plt.tight_layout()
# plt.show()

# diffusion distance comparison

In [ ]:
# # average of each row
ddm_multilayer_avg    = np.asarray(np.mean(ddm_both, axis=1)).ravel()
ddm_singlelayer_avg = np.asarray(np.mean(ddm_msigdb, axis=1)).ravel()
ddm_null_avg = np.asarray(np.mean(ddm_null, axis=1)).ravel()
ddm_multilayer_avg_DGIDB    = np.asarray(np.mean(ddm_both[:,dgidb_indices_agg], axis=1)).ravel()
ddm_singlelayer_avg_DGIDB = np.asarray(np.mean(ddm_msigdb[:,dgidb_indices], axis=1)).ravel()
ddm_null_avg_DGIDB = np.asarray(np.mean(ddm_null[:,dgidb_indices_agg], axis=1)).ravel()

In [ ]:
ddm_multilayer_avg_DGIDB_cropped = ddm_multilayer_avg_DGIDB[num_additional_rows:]
ddm_null_avg_DGIDB_cropped = ddm_null_avg_DGIDB[num_additional_rows:]

In [ ]:
ddm_multilayer_avg_DGIDB_cropped

In [ ]:
ddm_multilayer_avg_DGIDB_cropped.shape

In [ ]:
ddm_singlelayer_avg_DGIDB

In [ ]:
average_diff_dist_DGIDB_MSIGDB_singlelayer = np.asarray([ddm_singlelayer_avg[u] for u in dgidb_indices])
average_diff_dist_DGIDB_DGIDB_singlelayer = np.asarray([ddm_singlelayer_avg_DGIDB[u] for u in dgidb_indices])
average_diff_dist_MSIGDB_MSIGDB_singlelayer = np.asarray(ddm_singlelayer_avg)

In [ ]:
average_diff_dist_DGIDB_MSIGDB_multilayer = np.asarray([ddm_multilayer_avg[u] for u in dgidb_indices_agg])
average_diff_dist_DGIDB_DGIDB_multilayer = np.asarray([ddm_multilayer_avg_DGIDB[u] for u in dgidb_indices_agg])
average_diff_dist_MSIGDB_MSIGDB_multilayer = np.asarray(ddm_multilayer_avg[num_additional_rows:])

In [ ]:
# The null model shares the multilayer aggregate index space, so the same _agg indices apply.
average_diff_dist_DGIDB_MSIGDB_null = np.asarray([ddm_null_avg[u] for u in dgidb_indices_agg])
average_diff_dist_DGIDB_DGIDB_null = np.asarray([ddm_null_avg_DGIDB[u] for u in dgidb_indices_agg])
average_diff_dist_MSIGDB_MSIGDB_null = np.asarray(ddm_null_avg[num_additional_rows:])

In [ ]:
# setting universal font sizes
font_size = 25
tick_font_size = 16
plt.rcParams.update({
    "font.size": font_size,          
    "axes.titlesize": font_size,
    "axes.labelsize": font_size,
    "xtick.labelsize": tick_font_size,
    "ytick.labelsize": tick_font_size,
    "legend.fontsize": font_size,
    "figure.titlesize": font_size,
    "legend.loc": 'best'
})


In [ ]:
# DGIDB rows comparing to all genes
xaxis = range(len(ddm_multilayer_avg_DGIDB_cropped))
plt.figure(figsize=(10, 6))
plt.scatter(xaxis, ddm_multilayer_avg_DGIDB_cropped, label='Multilayer', color='blue')
plt.scatter(xaxis, ddm_null_avg_DGIDB_cropped, label='Null model', color='green')
plt.scatter(xaxis, ddm_singlelayer_avg_DGIDB, label='Single-layer', color='orange')
plt.yscale('log')
# legend placed outside of plot
plt.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
plt.savefig(f"../../Graphs/{DISEASE}/diff_dist_DGIDB_MSIGDB_comp.png", bbox_inches='tight')

In [ ]:
# DGIDB rows comparing to all genes
xaxis = range(len(average_diff_dist_DGIDB_MSIGDB_singlelayer))
plt.figure(figsize=(10, 6))
plt.scatter(xaxis, average_diff_dist_DGIDB_MSIGDB_multilayer, label='Multilayer', color='blue')
plt.scatter(xaxis, average_diff_dist_DGIDB_MSIGDB_null, label='Null model', color='green')
plt.scatter(xaxis, average_diff_dist_DGIDB_MSIGDB_singlelayer, label='Single-layer', color='orange')
plt.yscale('log')
# legend placed outside of plot
plt.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
plt.savefig(f"../../Graphs/{DISEASE}/diff_dist_DGIDB_MSIGDB_comp.png", bbox_inches='tight')

In [ ]:
# DGIDB rows comparing to DGIDB genes
xaxis = range(len(average_diff_dist_DGIDB_DGIDB_singlelayer))
plt.figure(figsize=(10, 6))
plt.scatter(xaxis, average_diff_dist_DGIDB_DGIDB_multilayer, label='Multilayer', color='blue')
plt.scatter(xaxis, average_diff_dist_DGIDB_DGIDB_null, label='Null model', color='green')
plt.scatter(xaxis, average_diff_dist_DGIDB_DGIDB_singlelayer, label='Single-layer', color='orange')
plt.yscale('log')
# legend placed outside of plot
plt.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
plt.savefig(f"../../Graphs/{DISEASE}/diff_dist_DGIDB_DGIDB_comp.png", bbox_inches='tight')

In [ ]:
# all rows comparing to all genes
xaxis = range(len(average_diff_dist_MSIGDB_MSIGDB_singlelayer))
plt.figure(figsize=(10, 6))
plt.scatter(xaxis, average_diff_dist_MSIGDB_MSIGDB_multilayer, label='Multilayer', color='blue')
plt.scatter(xaxis, average_diff_dist_MSIGDB_MSIGDB_null, label='Null model', color='green')
plt.scatter(xaxis, average_diff_dist_MSIGDB_MSIGDB_singlelayer, label='Single-layer', color='orange')
plt.yscale('log')
# legend placed outside of plot
plt.legend(bbox_to_anchor=(1.5, 1), loc='upper right')
plt.savefig(f"../../Graphs/{DISEASE}/diff_dist_MSIGDB_MSIGDB_comp.png", bbox_inches='tight')

In [ ]:
# DGIDB rows comparing to all genes - Histogram
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax in axes:
    ax.set_yscale("log")
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
    

axes[0].hist(average_diff_dist_DGIDB_MSIGDB_multilayer, bins=100, color='blue')
axes[0].set_title("Multi-layer")
axes[1].hist(average_diff_dist_DGIDB_MSIGDB_null, bins=100, color='green')
axes[1].set_title("Null model")
axes[2].hist(average_diff_dist_DGIDB_MSIGDB_singlelayer, bins=100, color='orange')
axes[2].set_title("Single-layer")

fig.supylabel("Frequency", x = 0.05, fontsize = 20)
fig.supxlabel("Average diffusion distance to DGIdb genes", y=0.01, fontsize = 20)  # changed: move shared x-label upward
fig.subplots_adjust(bottom=0.22, wspace=0.08)

plt.savefig(f"../../Graphs/{DISEASE}/diff_dist_DGIDB_MSIGDB_histogram.png", bbox_inches='tight')
plt.show()



In [ ]:
# DGIDB rows comparing to all genes - Histogram
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax in axes:
    ax.set_yscale("log")
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

axes[0].hist(average_diff_dist_MSIGDB_MSIGDB_multilayer, bins=100, color='blue')
axes[0].set_title("Multi-layer")

axes[1].hist(average_diff_dist_MSIGDB_MSIGDB_null, bins=100, color='green')
axes[1].set_title("Null model")

axes[2].hist(average_diff_dist_MSIGDB_MSIGDB_singlelayer, bins=100, color='orange')
axes[2].set_title("Single-layer")

fig.supylabel("Frequency", x = 0.05, fontsize = 20)
fig.supxlabel("Average diffusion distance to all genes", y=0.01, fontsize = 20)  # changed: move shared x-label upward
fig.subplots_adjust(bottom=0.22, wspace=0.08)

plt.savefig(f"../../Graphs/{DISEASE}/diff_dist_MSIGDB_MSIGDB_histogram.png", bbox_inches='tight')
plt.show()



In [ ]:
diff_dist_stat = {
    "DGIDB_MSIGDB_singlelayer": {"mean": float(average_diff_dist_DGIDB_MSIGDB_singlelayer.mean()), "var": float(average_diff_dist_DGIDB_MSIGDB_singlelayer.var())},
    "DGIDB_MSIGDB_multilayer": {"mean": float(average_diff_dist_DGIDB_MSIGDB_multilayer.mean()), "var": float(average_diff_dist_DGIDB_MSIGDB_multilayer.var())},
    "DGIDB_MSIGDB_null": {"mean": float(average_diff_dist_DGIDB_MSIGDB_null.mean()), "var": float(average_diff_dist_DGIDB_MSIGDB_null.var())},
    "DGIDB_DGIDB_singlelayer": {"mean": float(average_diff_dist_DGIDB_DGIDB_singlelayer.mean()), "var": float(average_diff_dist_DGIDB_DGIDB_singlelayer.var())},
    "DGIDB_DGIDB_multilayer": {"mean": float(average_diff_dist_DGIDB_DGIDB_multilayer.mean()), "var": float(average_diff_dist_DGIDB_DGIDB_multilayer.var())},
    "DGIDB_DGIDB_null": {"mean": float(average_diff_dist_DGIDB_DGIDB_null.mean()), "var": float(average_diff_dist_DGIDB_DGIDB_null.var())},
    "MSIGDB_MSIGDB_singlelayer": {"mean": float(average_diff_dist_MSIGDB_MSIGDB_singlelayer.mean()), "var": float(average_diff_dist_MSIGDB_MSIGDB_singlelayer.var())},
    "MSIGDB_MSIGDB_multilayer": {"mean": float(average_diff_dist_MSIGDB_MSIGDB_multilayer.mean()), "var": float(average_diff_dist_MSIGDB_MSIGDB_multilayer.var())},
    "MSIGDB_MSIGDB_null": {"mean": float(average_diff_dist_MSIGDB_MSIGDB_null.mean()), "var": float(average_diff_dist_MSIGDB_MSIGDB_null.var())},
}

In [ ]:
with open(f"../../Graphs/{DISEASE}/diff_dist_stat.json", "w") as f:
    json.dump(diff_dist_stat, f, indent=4)

In [ ]:
# plt.figure(figsize=(10, 6))
# plt.hist([ddm_msigdb_avg,ddm_both_avg_cropped], color=['orange', 'blue'], bins = 1000)
# # plt.xlabel("Ratio of Average Diffusion Distances (Multilayer / Single-layer)", fontsize=17)
# # plt.ylabel("Count", fontsize=17)
# plt.yscale('log')
# # legend placed outside of plot
# plt.legend(['Single-layer', 'Multilayer'], bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=17)

In [ ]:
# diff = ddm_both_avg_cropped / ddm_msigdb_avg

# plt.figure(figsize=(10, 6))
# plt.hist(diff, color='orange', bins = 1000)
# plt.axvline(1, color="black", linestyle="--", linewidth=2, label="baseline = 1")
# plt.xlabel("Ratio of Average Diffusion Distances (Multilayer / Single-layer)", fontsize=17)
# plt.ylabel("Count", fontsize=17)
# plt.yscale('log')
# # legend placed outside of plot